# NexaAssist: four-agent LangGraph lab

**Run All, offline.** This notebook imports the repository's actual graph, HTTP retrieval adapter, deterministic Validator, grounded answer chain and Action adapter. Only the classifier/model and external HTTP responses are faked. All external HTTP responses are mocked; this lab does not call Atlas, model providers or a live banking API. The current setup cell loads `.env` into Settings, so valid configuration values are needed to run it. These values must not be displayed or committed.

You will trace a policy answer; prepare and resume a callback; compare certificate, statement, KYC reminder and opt-in; reject a proposal; and verify safe failures. `INTEREST_CERTIFICATE` registers a request, not a generated PDF.

Use the repository `.venv` Python kernel in VS Code/Jupyter. Run cells in order; rerun from the beginning to reset the lab. In-memory checkpoints disappear on kernel restart. This is a graph-mechanics lab, **not production authentication or durable-run certification**.

**Configuration note:** The setup cell has been customized to use `load_dotenv` and environment values. This Markdown update preserves that executable setup. Reading configuration is distinct from using a live integration: the lab still injects fake model output and `httpx.MockTransport` clients.

In [1]:
from __future__ import annotations

import inspect
import json
import sys
from contextlib import asynccontextmanager
from copy import deepcopy
from datetime import date
from pathlib import Path
from uuid import uuid4

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "app/agents/orchestrator.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import httpx
from IPython.display import Markdown, display
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.runnables import RunnableLambda
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from app.agents.orchestrator import build_agent_graph
from app.chains.answer import build_answer_chain
from app.core.config import Settings
from app.models.agent import AgentRequest, AgentSubmissionRequest, IntentDecision
from app.models.answer import AnswerDraft
from app.models.validation import ValidationAccess
from app.services.agent_retrieval import retrieve_over_http
from app.services.service_requests import submit_service_request
from app.services.validation import OPT_IN_RULE, SERVICE_REQUEST_RULE, validate_service_request
from dotenv import load_dotenv
import os

load_dotenv(".env")


AS_OF = "2026-07-15"
OWNER = ValidationAccess(principal_id="lab-owner", customer_ids={"NXC-0010"})
# Explicit synthetic configuration; no Settings() call reads the real .env.
SETTINGS = Settings(
    _env_file=None,
    mongodb_uri=os.getenv("MONGODB_URI"),
    mongodb_db=os.getenv("MONGODB_DB"),
    mongodb_collection=os.getenv("MONGODB_COLLECTION"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    institution="NexaBank",
    jurisdiction="IN",
    confidentiality_level="internal",
)
display(Markdown("Repository imports loaded. External services remain mocked."))

Repository imports loaded. External services remain mocked.

## 1. Map the actual implementation

Four logical roles run inside one Agent Service. HTTP connects Streamlit → Agent FastAPI, Retriever → RAG API, and Validator/Action → bank. Graph-node handoffs are local async calls, not JSON-RPC.

The following cell derives current source lines rather than relying on old slide numbers. Read the referenced functions alongside the cells.

### The four agents at a glance

An **agent here is a logical role**, implemented by graph nodes and injected callables. It is not a separately deployed process. The Orchestrator uses a structured model for routing in production; the Validator and Action apply deterministic Python checks. The Retriever delegates policy search to the RAG API. A grounded answer is a supporting branch, not a fifth banking agent.

| Role | Receives | Main operation | Writes into graph state | Does not do |
|---|---|---|---|---|
| Orchestrator | `state["request"]`: `AgentRequest` | Screen unsafe input, classify intent, reconcile required scope, select a route | Resolved `request`, structured `decision` | Grant customer access or submit banking requests |
| Retriever | Resolved request and intent | Build a policy lookup and call existing POST `/retrieve` | `policy_request`, complete `retrieved` response | Read policy MongoDB directly from the agent or decide eligibility |
| Validator | Request, retained policy evidence and trusted customer access | Check applicable permission and read current customer/loan facts | `validation`, response and, when allowed, `proposal_hash` | Approve as a human or write to the bank |
| Action | Approved request, fresh allowed validation and saved operation key | Check the approved fingerprint and submit the exact proposal | Final `response` with submission receipt | Fulfill the callback, generate a certificate PDF or choose an arbitrary endpoint |

### Agent 1 — Orchestrator: decide which work is permitted to start

**Code:** `app/agents/orchestrator.py`, inner `orchestrator()` and `supported_action()`; production classification is composed in `app/services/agent.py:build_workflow`.

1. Read the typed `AgentRequest`. Structured fields such as customer, product and lookup date remain separate from natural-language `query`.
2. Check `unsafe_input(request)` before the classifier. A forbidden decision ends without retrieval or banking calls.
3. Call the injected classifier to obtain `IntentDecision`. The notebook's `classify()` is intentionally deterministic and recognizes only the lab examples; it does not measure the production model's routing accuracy.
4. Reconcile explicit request fields with model-extracted values. Conflicting values require clarification rather than silently replacing the user's scope.
5. Require the policy date and any required product; service requests additionally need customer, action and details. Certificate and opt-in requests need a loan ID; opt-in needs the home-loan product.
6. For supported actions, check trusted customer access and whether the required integration is connected **before retrieval**.

The output `decision.intent` selects the next node. A policy question goes to Retriever and then grounded answering. A permitted service request goes to Retriever and then Validator. Missing scope goes to `clarification`; forbidden input goes to `forbidden`. General customer-context requests have an unavailable-tool exit in this stage.

**Example:** `INTEREST_CERTIFICATE` without a loan ID produces `needs_clarification`. The graph should not try guessing a loan from the customer's records. Section 8 asserts that this case makes no HTTP calls.

### Agent 2 — Retriever: supply a dated evidence contract

**Code:** `app/agents/orchestrator.py:retrieve` and `app/services/agent_retrieval.py:retrieve_over_http`.

1. Convert the policy-relevant scope into `RetrieveRequest`. Customer ID, action and action details are excluded from the RAG payload.
2. For ordinary service requests, use the fixed operations-policy query for `POL-OPS-001` clause 5.1. Its lookup is bank-wide, so the retrieval product filter is cleared; the original product remains in `AgentRequest` for loan validation.
3. For prepayment opt-in, use the `CIR-2026-07` clause 5.2 lookup and retain the product scope.
4. Await `httpx.AsyncClient.post("retrieve", json=...)`, check HTTP status, parse the strict `RetrieveResponse`, and verify that its `as_of` equals the requested date.
5. Retain the whole response: ranked child text, complete parent text, citations, amendments and timings. Do not reduce it to an untraceable context string.

**Why:** A policy permission is only useful if its date, complete conditions and source identity survive the handoff. Section 10 demonstrates HTTP, schema and date failures. In the lab, `MockTransport` returns synthetic evidence while the actual adapter still performs these checks.

**Retrieval count:** A policy answer retrieves once and feeds retained evidence into its answer chain. An approved action retrieves once during preparation and again after approval to refresh its evidence. The second action lookup is intentional.

### Agent 3 — Validator: turn evidence and facts into an allowed or blocked proposal

**Code:** `app/services/validation.py:validate_service_request`, `validate_opt_in`, `_read_accounts`; graph wrapper `app/agents/orchestrator.py:validate`.

The Validator receives the resolved request, full evidence and server-issued `ValidationAccess`. It returns `ValidationResult`, whose decision is `allowed`, `blocked`, `insufficient_evidence` or `unavailable`.

For ordinary service requests it checks authorized customer access, the supported action and required details; matching lookup dates; the complete expected operations-policy rule and its published metadata; and a strictly parsed GET `/v1/customers/{id}/accounts` response. A selected loan must be owned by the customer, and a supplied product must match that loan.

For opt-in it instead checks complete circular evidence, an applicable date, product `NEXA-HL`, an owned `POL-HL-V3` / `NHL-2025-10` loan sanctioned before 2026-07-01, and an explicit not-already-opted-in status. Missing banking facts are not inferred by a model.

An allowed result carries the exact `ServiceRequestProposal` and supporting evidence. The graph stores it as `state["validation"]`, builds the response, and computes `proposal_hash` when execution is connected. A blocked or insufficient-evidence result does not continue to Action.

**Example:** A certificate request for a loan absent from the returned customer accounts is blocked. A valid certificate request becomes an allowed proposal; it still has not generated or requested anything at the bank's write endpoint.

### Human approval — the control between Validator and Action

**Code:** `app/agents/orchestrator.py:approval` and `revalidate`; production review boundary `app/services/agent_runs.py:AgentRunService.approve`.

`proposal_digest(request, validation)` hashes deterministic JSON containing the request and complete validation result, including the proposal and policy evidence. The approval node calls `interrupt({"proposal_hash": ..., "validation": ...})`. LangGraph checkpoints the state and returns control to the caller. Production `AgentRunService` reports this saved state as `pending_approval`; the inner response may still say `validated`.

The owner manually shares a saved run ID. A different authorized reviewer loads the proposal, reviews the evidence and submits a decision with the exact returned hash. Production checks reviewer permission, customer access, different identity, expiry and the pending hash, then records who decided and when. There is no automatic notification or reviewer-assignment queue in the current code.

`Command(resume=decision)` delivers the decision back to the paused `interrupt()`. Rejection ends without a banking POST. Approval stores `approved_hash`, then `revalidate()` refreshes policy evidence and customer facts. Changed evidence/proposal or a failed eligibility check blocks the write.

**Notebook boundary:** The direct `Command` calls simulate the decision payload; they do not authenticate a reviewer. `InMemorySaver` is sufficient for this demonstration but disappears with the kernel. Real named-human review, durable MongoDB state, expiry and leases belong to the FastAPI run service.

### Agent 4 — Action: execute only the exact approved operation

**Code:** `app/agents/orchestrator.py:action`; production execution closure in `app/services/agent_runs.py`; HTTP adapter `app/services/service_requests.py:submit_service_request`.

1. Require the Action callback, trusted access to the customer and an approved hash equal to the current request/validation digest.
2. In production, recheck current owner entitlements and the unchanged banking identity/destination, preserve the original operation key, and enforce the safe retry window.
3. Reconstruct the typed expected proposal. Require an allowed validation and exact equality between that proposal and the validation's proposal.
4. POST to the fixed `/v1/service-requests` endpoint with the saved `Idempotency-Key`. The banking client uses a server banking credential, distinct from owner/reviewer credentials.
5. Check HTTP status, strictly parse the receipt and verify matching customer, request type and loan ID. Return the submitted receipt together with policy evidence.

**Why:** Human approval must not become permission to execute a different payload, access another customer or create a new operation on every retry. An uncertain submission is reconciled by retrying the same saved run; a fresh run would have a different operation key.

In this lab the POST reaches only `MockTransport`. Its receipt dictionary simulates an idempotent bank; it does not prove the real provider's replay or durability guarantees. `submitted` plus `RECEIVED` means accepted, not fulfilled. A callback has not happened, a certificate has not been generated, and the mock opt-in flag is not changed.

### Shared state is the handoff, not a global chat history

| State key | Produced by | Consumed by |
|---|---|---|
| `request`, `decision` | Orchestrator | Routing and downstream nodes |
| `policy_request`, `retrieved` | Retriever | Answerer, Validator and fresh revalidation |
| `validation`, `proposal_hash` | Validator wrapper | Approval and Action |
| `approved_hash` | Approval node | Revalidation and Action |
| `idempotency_key` | Run service; lab initial state | Action adapter |
| `response` | Answer, safe-stop, Validator or Action node | API/UI or notebook display |

The explanatory diagram is a conceptual route map; Section 4 prints the actual compiled application's Mermaid topology.

```text
Policy: Orchestrator → Retriever → grounded_answer → END
Action: Orchestrator → supported_action → Retriever → Validator
                                                    ↓ allowed
                                                 approval pause
                                                    ↓ approve
                                                 revalidate
                                                    ↓ still allowed
                                                   Action → END
```


In [2]:
symbols = [
    build_agent_graph,
    retrieve_over_http,
    validate_service_request,
    build_answer_chain,
    submit_service_request,
]
for symbol in symbols:
    path = Path(inspect.getsourcefile(symbol)).relative_to(ROOT)
    line = inspect.getsourcelines(symbol)[1]
    display(Markdown(f"- `{path}:{line}` — `{symbol.__name__}`"))

- `app/agents/orchestrator.py:60` — `build_agent_graph`

- `app/services/agent_retrieval.py:10` — `retrieve_over_http`

- `app/services/validation.py:56` — `validate_service_request`

- `app/chains/answer.py:26` — `build_answer_chain`

- `.venv/lib/python3.14/site-packages/ddtrace/_trace/tracer.py:18` — `submit_service_request`

## 2. Synthetic evidence and current customer facts

The policy fixture uses the exact rule constants expected by the real Validator. It is a controlled teaching fixture, **not independent proof of corpus quality**. The amendment retains a separate citation and complete parent text. Banking facts represent one owned pre-July-2026 home loan.

`as_of` is a policy lookup date, not a certificate financial-year field.

In [3]:
def hit(doc_id, clause, effective_date, text):
    return {
        "rank": 1,
        "score": 0.9,
        "text": text,
        "parent_text": text,
        "citation": {
            "doc_id": doc_id,
            "title": "Synthetic training policy",
            "policy_version": "1.0",
            "clause": clause,
            "effective_date": effective_date,
        },
    }


POLICY = {
    "as_of": AS_OF,
    "results": [hit("POL-OPS-001", "5.1", "2025-01-01", SERVICE_REQUEST_RULE)],
    "amendments": [hit("CIR-2026-07", "5.2", "2026-07-01", OPT_IN_RULE)],
    "timings": {"search_ms": 1, "parent_ms": 1, "total_ms": 2},
}
ACCOUNTS = {
    "customer_id": "NXC-0010",
    "accounts": [],
    "loans": [
        {
            "loan_account_id": "NXL-HL-0010",
            "product_id": "NEXA-HL",
            "policy_doc": "POL-HL-V3",
            "version_id": "NHL-2025-10",
            "sanction_date": "2025-12-12",
            "circ_2026_07_part_prepay_opt_in": False,
        }
    ],
}

## 3. Replace only the model and network boundaries

The classifier below returns a structured decision without making a model call. It deliberately recognizes only this lab's examples; it is not a replacement production classifier.

The fake chat model returns a draft through a Runnable, while the **real answer chain** still validates quote/citation grounding. The HTTP mock raises on unknown destinations and never falls through to a real network connection.

In [4]:
def classify(request):
    intent = (
        "forbidden"
        if "approve my loan" in request.query.lower()
        else "supported_action"
        if request.action is not None
        else "policy_question"
    )
    return IntentDecision(
        intent=intent,
        product_id=None,
        as_of=None,
        customer_id=None,
        action=None,
        product_required=False,
        missing_fields=[],
    )


class StructuredFake(FakeListChatModel):
    def with_structured_output(self, *args, **kwargs):
        return RunnableLambda(
            lambda _: AnswerDraft(
                status="answered",
                answer="Existing borrowers may opt in through a service request [1].",
                evidence=[{"evidence_id": 1, "quote": OPT_IN_RULE}],
            )
        )


MODEL = StructuredFake(responses=["unused: structured output is supplied locally"])


@asynccontextmanager
async def lab(*, fault=None, access=OWNER, bank_facts=None):
    events = []
    facts = deepcopy(bank_facts or ACCOUNTS)
    receipts = {}

    def respond(request):
        events.append({"method": request.method, "path": request.url.path})
        if request.url.host == "rag.test" and request.url.path == "/retrieve":
            if fault == "http":
                return httpx.Response(503, json={"detail": "Synthetic upstream failure"})
            if fault == "schema":
                return httpx.Response(200, json={"unexpected": "body"})
            body = deepcopy(POLICY)
            body["as_of"] = json.loads(request.content)["as_of"]
            if fault == "date":
                body["as_of"] = "2025-01-01"
            return httpx.Response(200, json=body)
        if request.url.host == "bank.test" and request.method == "GET":
            assert request.url.path == "/v1/customers/NXC-0010/accounts"
            return httpx.Response(200, json=facts)
        if request.url.host == "bank.test" and request.method == "POST":
            assert request.url.path == "/v1/service-requests"
            body = json.loads(request.content)
            key = request.headers["Idempotency-Key"]
            if key not in receipts:
                receipts[key] = {
                    "service_request_id": f"LAB-SR-{len(receipts) + 1:03}",
                    "customer_id": body["customer_id"],
                    "request_type": body["request_type"],
                    "loan_account_id": body.get("loan_account_id"),
                    "status": "RECEIVED",
                    "created_at": "2026-07-15T10:00:00+05:30",
                }
            return httpx.Response(201, json=receipts[key])
        raise AssertionError(f"Unexpected mocked HTTP request: {request.method} {request.url.path}")

    transport = httpx.MockTransport(respond)
    async with (
        httpx.AsyncClient(
            base_url="https://rag.test/", transport=transport, timeout=5, follow_redirects=False
        ) as rag,
        httpx.AsyncClient(
            base_url="https://bank.test/", transport=transport, timeout=5, follow_redirects=False
        ) as bank,
    ):

        async def retrieve(request):
            return await retrieve_over_http(request, rag)

        async def answer(request, evidence):
            chain = build_answer_chain(RunnableLambda(lambda _: evidence), MODEL, SETTINGS)
            return await chain.ainvoke(request)

        async def validate(request, evidence):
            return await validate_service_request(request, evidence, bank, access)

        async def execute(request, validation, key):
            typed = AgentSubmissionRequest.model_validate(request.model_dump())
            return await submit_service_request(typed, validation, bank, access, key)

        graph = build_agent_graph(
            RunnableLambda(classify),
            RunnableLambda(retrieve),
            answer,
            validator=validate,
            access=access,
            actioner=execute,
            checkpointer=InMemorySaver(),
        )
        yield graph, events


def run_config():
    return {"configurable": {"thread_id": uuid4().hex}, "recursion_limit": 12}


def service_request(kind="CALLBACK", **changes):
    body = {
        "query": f"Prepare a {kind} service request.",
        "as_of": AS_OF,
        "customer_id": "NXC-0010",
        "product_id": "NEXA-HL",
        "action": "createServiceRequest",
        "action_details": {
            "request_type": kind,
            "loan_account_id": "NXL-HL-0010",
            "notes": "Synthetic notebook demonstration.",
        },
    }
    body.update(changes)
    return AgentRequest.model_validate(body)

## 4. Inspect the real graph topology

The Mermaid text comes from the compiled application graph. It can be pasted into a Mermaid viewer; this cell does not call an online diagram renderer. Notice clarification/forbidden exits, the policy-answer branch and approval/revalidate/action nodes.

### Where `graph.compile()` happens

`build_agent_graph()` creates a `StateGraph(AgentState)`, registers nodes using `add_node`, and connects them with `add_edge` / `add_conditional_edges`. Its final statement is:

```python
return graph.compile(checkpointer=checkpointer)
```

The `graph` returned by the notebook's `lab()` is therefore already a `CompiledStateGraph`. Compilation prepares the executable routing/checkpoint machinery; it does not classify a query or call a tool. `await graph.ainvoke(initial_state, config)` runs it and returns state. `await graph.aget_state(config)` reads the saved checkpoint. `graph.get_graph().draw_mermaid()` describes its topology, not an execution result.

The same `configurable.thread_id` identifies the saved run during preparation, checkpoint inspection and resume. New examples use fresh IDs to avoid accidentally resuming an earlier demonstration. Section 5 displays a policy response; Section 6 displays the callback interrupt, HTTP trace and receipt; Section 7 displays all five service-request results.


In [5]:
async with lab() as (graph, events):
    display(Markdown("```mermaid\n" + graph.get_graph().draw_mermaid() + "\n```"))
    assert not events

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrator(orchestrator)
	retriever(retriever)
	grounded_answer(grounded_answer)
	clarification(clarification)
	forbidden(forbidden)
	customer_context(customer_context)
	supported_action(supported_action)
	validator(validator)
	approval(approval)
	revalidate(revalidate)
	action(action)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrator;
	approval -. &nbsp;end&nbsp; .-> __end__;
	approval -.-> revalidate;
	orchestrator -. &nbsp;missing_information&nbsp; .-> clarification;
	orchestrator -.-> customer_context;
	orchestrator -.-> forbidden;
	orchestrator -. &nbsp;policy_question&nbsp; .-> retriever;
	orchestrator -.-> supported_action;
	retriever -.-> grounded_answer;
	retriever -.-> validator;
	revalidate -. &nbsp;end&nbsp; .-> __end__;
	revalidate -.-> action;
	supported_action -. &nbsp;end&nbsp; .-> __end__;
	supported_action -. &nbsp;retrieve&nbsp; .-> retriever;
	validator -. &nbsp;end&nbsp; .-> __end__;
	validator -.-> approval;
	action --> __end__;
	clarification --> __end__;
	customer_context --> __end__;
	forbidden --> __end__;
	grounded_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 5. Policy question: retrieve once, then grounded answer

The actual answer chain gets a Runnable over the retained response, not another HTTP Retriever. Complete amendments are ordered first by `RetrieveResponse.evidence`, so evidence ID 1 references the circular. Exact quote validation is still performed by application code.

In [ ]:
async with lab() as (graph, events):
    state = await graph.ainvoke(
        {
            "request": AgentRequest(
                query="How may existing NexaHome borrowers opt in?",
                as_of=date.fromisoformat(AS_OF),
                product_id="NEXA-HL",
            )
        },
        run_config(),
    )
    assert state["response"].status == "answered"
    assert events == [{"method": "POST", "path": "/retrieve"}]
    assert state["response"].answer.citations[0].quote == OPT_IN_RULE
    display(state["response"].model_dump(mode="json"))

## 6. Callback: inspect the saved approval pause

`ainvoke` returns an interrupt; it does not keep the HTTP request open waiting for a person. The production run service translates this checkpoint into `pending_approval`. The graph's inner response is still `validated`. The run-level status and inner validation status are different concepts.

No banking POST may happen before resume. The same graph/checkpointer and thread configuration must be retained.

In [ ]:
async with lab() as (callback_graph, callback_events):
    config = run_config()
    request = service_request(
        "CALLBACK",
        product_id=None,
        action_details={"request_type": "CALLBACK", "notes": "Please arrange a callback."},
    )
    prepared = await callback_graph.ainvoke(
        {"request": request, "idempotency_key": "lab-callback-operation"}, config
    )
    checkpoint = await callback_graph.aget_state(config)
    assert checkpoint.next == ("approval",)
    assert prepared["__interrupt__"]
    assert prepared["response"].status == "validated"
    assert not any(e["path"] == "/v1/service-requests" for e in callback_events)
    assert prepared["retrieved"].amendments[0].parent_text == OPT_IN_RULE
    display(
        {
            "waiting_at": checkpoint.next,
            "proposal_hash": prepared["proposal_hash"],
            "validation": prepared["validation"].model_dump(mode="json"),
        }
    )

    # Simulated decision payload: this is NOT production reviewer authentication.
    decision = {"decision": "approve", "proposal_hash": prepared["proposal_hash"]}
    completed = await callback_graph.ainvoke(Command(resume=decision), config)
    assert completed["response"].status == "submitted"
    assert sum(e["path"] == "/retrieve" for e in callback_events) == 2
    assert sum(e["path"] == "/v1/service-requests" for e in callback_events) == 1
    display(callback_events)
    display(completed["response"].submission.model_dump(mode="json"))

The extra retrieval after approval is intentional: the action path refreshes policy and bank facts before writing. This does not contradict the one-retrieval policy-answer path. A `RECEIVED` receipt is acceptance, not callback completion.

**Trust boundary:** Direct `Command(resume=...)` is a mechanics demonstration. Production callers must use `/agent/runs/{id}/approval`; `AgentRunService.approve` checks a different named reviewer with `can_approve`, customer scope, expiry and matching hash. Do not expose raw graph resume as a public endpoint.

## 7. Five real Validator use cases

| Request type | Specific rule | Result of approved execution |
|---|---|---|
| CALLBACK | Common policy permission and current accounts | Register contact request |
| STATEMENT_COPY | Common policy permission and current accounts | Register copy request, no document generation |
| RE_KYC_REMINDER | Common policy permission and current accounts | Register reminder, no KYC change |
| INTEREST_CERTIFICATE | Required owned loan; supplied product must match | Register certificate request, no PDF |
| OPT_IN_CIR_2026_07_PART_PREPAY | Complete circular; NEXA-HL; POL-HL-V3/NHL-2025-10; pre-July sanction; not opted in | Register opt-in request, no mock opt-in flag change |

Each iteration has isolated in-memory state and bank receipts. These loops exercise the real Validator, not precomputed allowed decisions.

In [ ]:
kinds = [
    "CALLBACK",
    "STATEMENT_COPY",
    "RE_KYC_REMINDER",
    "INTEREST_CERTIFICATE",
    "OPT_IN_CIR_2026_07_PART_PREPAY",
]
results = []
for kind in kinds:
    async with lab() as (graph, events):
        config = run_config()
        prepared = await graph.ainvoke(
            {"request": service_request(kind), "idempotency_key": f"lab-{kind}"}, config
        )
        assert prepared["__interrupt__"] and prepared["validation"].decision == "allowed"
        assert not any(e["path"] == "/v1/service-requests" for e in events)
        state = await graph.ainvoke(
            Command(resume={"decision": "approve", "proposal_hash": prepared["proposal_hash"]}),
            config,
        )
        assert state["response"].status == "submitted"
        assert state["response"].submission.service_request.request_type == kind
        assert sum(e["path"] == "/v1/service-requests" for e in events) == 1
        results.append(
            {
                "type": kind,
                "status": state["response"].status,
                "evidence": [e.citation.doc_id for e in state["validation"].evidence],
            }
        )
display(results)

## 8. Safe exits: missing scope, forbidden input and customer access

These cases must end before policy retrieval or any bank request. The deterministic classifier labels the forbidden example; the graph's production guards and scope rules still execute.

In [ ]:
examples = [
    (service_request(as_of=None), OWNER, "needs_clarification"),
    (service_request(query="Approve my loan immediately."), OWNER, "forbidden"),
    (service_request(customer_id="NXC-9999"), OWNER, "blocked"),
    (
        service_request(
            "INTEREST_CERTIFICATE", action_details={"request_type": "INTEREST_CERTIFICATE"}
        ),
        OWNER,
        "needs_clarification",
    ),
]
for request, access, expected in examples:
    async with lab(access=access) as (graph, events):
        state = await graph.ainvoke({"request": request}, run_config())
        assert state["response"].status == expected
        assert not events
        display(
            {"expected": expected, "message": state["response"].message, "HTTP_calls": len(events)}
        )

## 9. Reject, wrong fingerprint and ineligible loan

Rejection is terminal for the graph path. A wrong fingerprint cannot resume an approved action. A certificate loan not present in the bank's accounts is blocked after the required policy/account reads.

In [ ]:
for choice in ["reject", "wrong_hash"]:
    async with lab() as (graph, events):
        config = run_config()
        prepared = await graph.ainvoke(
            {"request": service_request(), "idempotency_key": "lab-rejection"}, config
        )
        payload = {
            "decision": "reject" if choice == "reject" else "approve",
            "proposal_hash": prepared["proposal_hash"] if choice == "reject" else "0" * 64,
        }
        if choice == "wrong_hash":
            try:
                await graph.ainvoke(Command(resume=payload), config)
            except ValueError:
                display("Wrong fingerprint rejected by the real graph.")
            else:
                raise AssertionError("Wrong fingerprint unexpectedly accepted")
        else:
            state = await graph.ainvoke(Command(resume=payload), config)
            assert state["response"].status == "rejected"
        assert not any(e["path"] == "/v1/service-requests" for e in events)

async with lab(bank_facts={**deepcopy(ACCOUNTS), "loans": []}) as (graph, events):
    state = await graph.ainvoke({"request": service_request("INTEREST_CERTIFICATE")}, run_config())
    assert state["response"].status == "blocked"
    assert not any(e["path"] == "/v1/service-requests" for e in events)
    display(state["response"].message)

## 10. Upstream HTTP, schema and lookup-date failures

The HTTP adapter must fail rather than answer or submit with invalid evidence. At this direct graph layer exceptions propagate. Production routers/services translate them into safe API responses; they must not expose raw upstream diagnostics.

In [ ]:
for fault in ["http", "schema", "date"]:
    async with lab(fault=fault) as (graph, events):
        try:
            await graph.ainvoke({"request": service_request()}, run_config())
        except (httpx.HTTPStatusError, ValueError) as exc:
            display({"fault": fault, "exception_type": type(exc).__name__})
        else:
            raise AssertionError(f"Invalid upstream evidence accepted: {fault}")
        assert not any(e["path"] == "/v1/service-requests" for e in events)

## 11. From notebook to Streamlit and FastAPI

| Operation | Existing endpoint | Caller / meaning |
|---|---|---|
| Prepare | POST `/agent/runs` | Owner bearer credential; returns 201 with the run's current business status |
| Inspect | GET `/agent/runs/{id}` | Owner or entitled reviewer; read the saved proposal/evidence |
| Decide | POST `/agent/runs/{id}/approval` | Different entitled reviewer; `decision` + exact `proposal_hash` |
| Retry uncertain write | POST `/agent/runs/{id}/retry` | Same saved approved operation; no fresh key |
| Standalone agent | POST `/agent` | Policy answering or action validation; no durable approval execution |

Current contracts use flat Pydantic responses and FastAPI `detail` errors. Do not substitute generic API-design examples or invent `/api/v1` routes. Named credentials/customer scope failures use 403, inaccessible saved runs 404, stale or conflicting operations 409, request schema failures 422 and unavailable services 503. HTTP success does not mean banking fulfillment.

In Streamlit: Service requests → owner credential → select customer/type/date/loan → Prepare for approval → save run ID → switch to different reviewer → load run → inspect evidence → approve. Changing credentials clears the displayed proposal; it does not delete the saved run.

There are deliberately **no live HTTP cells** here. Use Swagger manually if you want to exercise configured services and actual synthetic banking writes.

In [ ]:
from app.api.agent_runs import approve, read, retry, start
from app.models.agent_run import proposal_digest
from app.services.agent_runs import AgentRunService

for symbol in [start, read, approve, retry, AgentRunService.approve, proposal_digest]:
    path = Path(inspect.getsourcefile(symbol)).relative_to(ROOT)
    display(Markdown(f"`{path}:{inspect.getsourcelines(symbol)[1]}` — `{symbol.__qualname__}`"))
display(
    Markdown("**Run All complete:** real graph and adapters executed against local mocks only.")
)

## 12. What this proves, and what it does not

The assertions verify real graph routing, single retrieval for a policy answer, complete amendment/citation handoff, approval interrupt/resume, all five real Validator paths, rejected/missing/unauthorized scopes, loan ownership failure and invalid upstream evidence. Banking writes go only to MockTransport.

Production additionally requires named-principal authentication, separate-reviewer checks, MongoDB checkpoint durability, expiry and leases, banking identity consistency, safe uncertain-write reconciliation and configured provider/Atlas retrieval. `InMemorySaver` and the fake receipt dictionary do not prove those guarantees, model quality, production latency or an end-to-end UI deployment.

Suggested exercises: remove the certificate loan ID; mismatch product; change sanction date; set opt-in to true; remove circular amendments; reject a callback; change evidence between preparation and approval and observe the digest guard. Re-run the harness from its definition after editing a fixture.

Further regression references: `tests/test_agent_action_graph.py`, `tests/test_agent_runs.py`, `tests/test_agent_run_auth.py`, `tests/test_agent_persistence.py`, `tests/test_validation.py`, `tests/test_action_receipt.py`.